# Lesson 4g — Spelling from scratch: a *character-level* model (nursery rhymes)

The **runnable companion** to the *"from memorizer to general-purpose"* section of the Lesson 4f webpage.

Lesson 4f built a **next-WORD** model: it knew **31 whole words** and predicted the next *word*. It writes neat
sentences, but it can only ever use those 31 words — type `puppy` and it has no box for it. That is *not* how
ChatGPT is general purpose.

This notebook makes the real leap: a **next-CHARACTER** model. Its whole alphabet is ~25 letters, and it predicts
the next *letter*, one at a time. Out of a handful of letters it can spell **any** word — so it can read and write
things it never saw in training. That is the GPT recipe in miniature.

### How it differs from Lesson 4f

| | 4f — next **word** | 4g — next **character** (this) |
|---|---|---|
| token = | a whole word | a single character |
| vocabulary | 31 fixed words | ~25 letters |
| can handle a new word? | ❌ no box for it | ✅ spell it from letters |
| training data | 29 short sentences | 13 nursery rhymes (one long stream) |
| how we cut training examples | pad each sentence, predict next word | slide a window over the text, predict next char |
| special tokens | `<bos> <eos> <pad>` | none — just raw text |
| what it learns | which word follows which | how to **spell**, then how to **rhyme** |

Everything else — embed → attention → feed-forward → head → softmax, trained with a causal mask — is **identical**.
Only the size of a "token" changed.

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)

## 1. The corpus — nursery rhymes as one long stream

A next-word model wants a *list of sentences*. A character model just wants **text** — one long string. We'll use
13 classic rhymes (~1,900 characters). They rhyme and repeat, so a tiny model can learn to spell and echo them.

In [2]:
CORPUS = 'twinkle twinkle little star\nhow i wonder what you are\nup above the world so high\nlike a diamond in the sky\ntwinkle twinkle little star\nhow i wonder what you are\n\nhumpty dumpty sat on a wall\nhumpty dumpty had a great fall\nall the kings horses and all the kings men\ncould not put humpty together again\n\nbaa baa black sheep have you any wool\nyes sir yes sir three bags full\none for the master and one for the dame\nand one for the little boy who lives down the lane\n\njack and jill went up the hill\nto fetch a pail of water\njack fell down and broke his crown\nand jill came tumbling after\n\nhickory dickory dock\nthe mouse ran up the clock\nthe clock struck one the mouse ran down\nhickory dickory dock\n\nthe itsy bitsy spider climbed up the water spout\ndown came the rain and washed the spider out\nout came the sun and dried up all the rain\nand the itsy bitsy spider climbed up the spout again\n\nmary had a little lamb its fleece was white as snow\nand everywhere that mary went the lamb was sure to go\nit followed her to school one day which was against the rule\nit made the children laugh and play to see a lamb at school\n\nlittle miss muffet sat on a tuffet\neating her curds and whey\nalong came a spider who sat down beside her\nand frightened miss muffet away\n\nrow row row your boat gently down the stream\nmerrily merrily merrily merrily life is but a dream\n\nold macdonald had a farm\nand on his farm he had a cow\nwith a moo moo here and a moo moo there\nhere a moo there a moo everywhere a moo moo\n\nthe wheels on the bus go round and round\nround and round round and round\nthe wheels on the bus go round and round\nall through the town\n\nrain rain go away come again another day\nlittle children want to play rain rain go away\n\nhey diddle diddle the cat and the fiddle\nthe cow jumped over the moon\nthe little dog laughed to see such sport\nand the dish ran away with the spoon\n'

print(f"{len(CORPUS)} characters, {CORPUS.count(chr(10))} lines")
print(CORPUS[:120], '...')

1861 characters, 62 lines
twinkle twinkle little star
how i wonder what you are
up above the world so high
like a diamond in the sky
twinkle twink ...


## 2. Vocabulary — characters, not words  *(the key change)*

In 4f every distinct **word** got an id. Here every distinct **character** gets an id — letters, the space, the
newline. That's the whole dictionary. Notice there are no `<bos>`/`<eos>`/`<pad>` tokens: we don't pad anything,
we just slide a window across the raw text.

Because the alphabet is tiny and *closed over letters*, the model can spell **any** word built from these
characters — even one that never appears in the rhymes.

In [3]:
chars = sorted(set(CORPUS))
V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}      # char  -> id
itos = {i: c for c, i in stoi.items()}          # id    -> char

# the whole text as one long tensor of ids
data = torch.tensor([stoi[c] for c in CORPUS], dtype=torch.long)
T = 64                                           # context window (chars the model sees at once)

print(f"V = {V} characters: {chars}")
print(f"data tensor: {tuple(data.shape)}")

V = 25 characters: ['\n', ' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y']
data tensor: (1861,)


## 3. The model — a character Transformer

Almost the same class as 4f's `NextWord`. Two cosmetic differences:

- the embedding table is `V` **characters** wide instead of `V` words,
- we use `nn.TransformerEncoder(..., num_layers=n)` so we can **stack** layers later.

The causal mask is the same trick: each position may only look left, so predicting the next character is honest.

In [4]:
class CharGPT(nn.Module):
    def __init__(self, d=64, nhead=4, ff=128, nlayers=2):
        super().__init__()
        self.tok = nn.Embedding(V, d)            # char id  -> vector
        self.pos = nn.Embedding(T, d)            # position -> vector
        layer = nn.TransformerEncoderLayer(d, nhead, ff, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, num_layers=nlayers)
        self.head = nn.Linear(d, V)              # vector   -> score per character

    def forward(self, x):
        t = x.size(1)
        pos = torch.arange(t).unsqueeze(0)
        h = self.tok(x) + self.pos(pos)
        mask = nn.Transformer.generate_square_subsequent_mask(t)   # no peeking right
        return self.head(self.enc(h, mask=mask))

## 4. Training data — slide a window, predict the next char

In 4f we padded each sentence and predicted the next *word*. Here we grab a **random window** of `T` characters
as the input, and the *same window shifted one step* as the target. So for `...the cat...` the model learns
`t`→`h`, `h`→`e`, `e`→` `, and so on. Predicting the next character, everywhere, at once.

In [5]:
def get_batch(bs=32):
    ix = torch.randint(0, len(data) - T - 1, (bs,))
    xb = torch.stack([data[i:i+T]     for i in ix])   # window
    yb = torch.stack([data[i+1:i+T+1] for i in ix])   # same window, shifted by 1
    return xb, yb

xb, yb = get_batch(1)
print('input :', ''.join(itos[i] for i in xb[0][:40].tolist()))
print('target:', ''.join(itos[i] for i in yb[0][:40].tolist()))

input : go
it followed her to school one day whi
target: o
it followed her to school one day whic


## 5. Train — and watch it learn to spell

This is the best part. We train for 4,000 steps and, at a few checkpoints, ask the half-trained model to write
110 characters starting from `"the "`. Watch the output climb: **noise → letter clumps → real words → whole
rhymes.** (Sampling uses a little randomness, so it explores rather than repeating one safe letter.)

In [6]:
@torch.no_grad()
def sample(model, seed='the ', n=110, temp=0.7):
    model.eval()
    ids = [stoi[c] for c in seed if c in stoi] or [stoi[' ']]
    for _ in range(n):
        logits = model(torch.tensor([ids[-T:]]))[0, -1] / temp
        p = F.softmax(logits, -1)
        ids.append(int(torch.multinomial(p, 1)))
    return ''.join(itos[i] for i in ids)

def train(nlayers=2, steps=4000, show=False):
    torch.manual_seed(0)
    model = CharGPT(nlayers=nlayers)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    checkpoints = {0, 150, 400, 1000, 2500, steps}
    for step in range(steps + 1):
        if show and step in checkpoints:
            print(f'--- step {step:>4} ---')
            print(sample(model)[:110])
            print()
            model.train()
        if step == steps:
            break
        xb, yb = get_batch(32)
        loss = F.cross_entropy(model(xb).reshape(-1, V), yb.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        xb, yb = get_batch(128)
        final = F.cross_entropy(model(xb).reshape(-1, V), yb.reshape(-1)).item()
    return model, sum(p.numel() for p in model.parameters()), final

model, params_2L, loss_2L = train(nlayers=2, steps=4000, show=True)
print(f'2-layer model: {params_2L:,} parameters, final loss {loss_2L:.3f}')

--- step    0 ---
the jtrnuyrilvpkfmeyhbockubplgtrkhwujmtu
k gsjhcbpeuthciiflb iyi
eoyjhhrrf
wpyugyeryedijhwp kraacbvauejinefpe




--- step  150 ---
the ma wint

up tchithittd aithain w are the le t t che sankoon sst tlithee fe spiche l

try pitttstt piditts 



--- step  400 ---
the a moo moo there
here a a moo moo here and and the laughed and and plle to the din

awin the the the lane
c



--- step 1000 ---
the clock struck one the mouse ran down
hickory dickory dock
the itsy bitsy spider climbed up the water spout




--- step 2500 ---
the sun and dried up all the rain
and the itsy bitsy spider climbed up the water spout
down came the rain and 



--- step 4000 ---
the sun and dried up all the rain
and the itsy bitsy spider climbed up the water spout
down came the rain and 

2-layer model: 74,265 parameters, final loss 0.104


By the last checkpoint the model spells correctly and reproduces whole verses — having only ever seen **letters**,
never a single whole word. It learned English spelling and the shape of these rhymes from raw characters.

## 6. Generate — give it the start of any rhyme

The ChatGPT loop, one **character** at a time: predict next char, append, repeat.

In [7]:
for seed in ['twinkle twinkle ', 'the cat ', 'jack and jill ', 'humpty ']:
    print(repr(seed), '->')
    print(sample(model, seed, n=100))
    print()

'twinkle twinkle ' ->
twinkle twinkle little star
how i wonder what you are
up above the world so high
like a diamond in the sky
twinkle t

'the cat ' ->
the cat and the fiddle
the cow jumped over the moon
the little dog laughed to see such sport
and the dish ra

'jack and jill ' ->


jack and jill went up the hill
to fetch a pail of water
jack fell down and broke his crown
and jill came tumbling 

'humpty ' ->
humpty dumpty had a great fall
all the kings horses and all the kings men
could not put humpty together aga



## 7. Why this is "general purpose": the out-of-vocabulary wall

Here's the difference made concrete. The 4f next-word model had a fixed 31-word dictionary. If a word isn't in
it, there is literally **no id** for it — the model can't even read the input. The character model spells the
same word from letters it already knows.

In [8]:
word_vocab = ['box','boy','cat','couch','dog','door','girl','he','i','in','likes','mat','on',
              'opened','park','play','plays','ran','run','sat','she','sleep','sleeps','store',
              'the','to','we','went']   # the 31-word model's entire dictionary (minus specials)

for w in ['cat', 'puppy', 'banana', 'dinosaur']:
    a = 'OK' if w in word_vocab else 'CANT READ IT'
    b = 'can spell it' if all(c in stoi for c in w) else 'missing a letter'
    print(f'{w:9s}  whole-word: {a:14s}  character: {b}')

cat        whole-word: OK              character: can spell it
puppy      whole-word: CANT READ IT    character: can spell it
banana     whole-word: CANT READ IT    character: can spell it
dinosaur   whole-word: CANT READ IT    character: can spell it


`cat` is the only one of those the whole-word model can handle. The character model spells all four — that open
vocabulary is exactly what "general purpose" means. (Real GPTs use **subword** pieces: a middle ground that keeps
common words whole but can still spell anything from fragments.)

## 8. Now stacking layers finally *helps*

In Lesson 4f's webpage we saw a 2nd layer do **nothing** — the 29-sentence toy was too easy. Spelling real rhymes
from characters is genuinely harder, so depth now pays off. Same experiment here:

In [9]:
m1, p1, l1 = train(nlayers=1, steps=4000)
print(f'1 layer : {p1:,} params, final loss {l1:.3f}')
print(f'2 layers: {params_2L:,} params, final loss {loss_2L:.3f}')
print(f'-> more params AND lower loss: depth helps when the task is hard enough.')

1 layer : 40,793 params, final loss 0.157
2 layers: 74,265 params, final loss 0.104
-> more params AND lower loss: depth helps when the task is hard enough.


## 9. Your turn — experiments

1. **Feed it your own text.** Replace `CORPUS` with a few of your favourite songs or a short story, re-run from
   cell 1, and watch it learn *that* style. (Keep it lowercase to keep the alphabet small.)
2. **Temperature.** In `sample(...)`, try `temp=0.3` (safe, repetitive) vs `temp=1.2` (wild, more typos). This is
   the same creativity dial as the webpage slider.
3. **Make it bigger.** Try `CharGPT(d=96, nlayers=3)`. Does the loss drop further? Does it overfit (memorise)?
4. **Go smaller.** `CharGPT(d=16, nlayers=1)` — how garbled does spelling get?
5. **Add capitals/punctuation** to the corpus and see the alphabet `V` grow. Each new symbol is a new id the model
   now has to learn.